# Introducción a la Ciencia de Datos: Tarea 2

Este notebook contiene el código de base para realizar la Tarea 2 del curso. Es la continuación de la Tarea 1, por lo que se utilizarán los mismos datos y se puede reutilizar cualquier parte del código de dicha tarea.

Puede copiar este notebook en su propio repositorio y trabajar sobre el mismo.
Las **instrucciones para ejecutar el notebook** están en la [página inicial del repositorio](https://gitlab.fing.edu.uy/maestria-cdaa/intro-cd).

Se utiliza el lenguaje Python y las librerías Pandas y scikit-learn. Para esta tarea se recomienda consultar la sección [Extracting features from text files](https://scikit-learn.org/stable/tutorial/text_analytics/working_with_text_data.html) de la documentación oficial de scikit-learn.

Recuerde que **se espera que no sea necesario revisar el código para corregir la tarea**, ya que todos los resultados y análisis relevantes deberían estar en el **informe en formato PDF**.

## Cargar bibliotecas (dependencias)
Recuerde instalar los requerimientos (`requirements.txt`) en el mismo entorno donde está ejecutando este notebook.

In [ ]:
from time import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from datasets import load_dataset

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import PCA
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
)

# Agregue aqui el resto de las librerias que necesite
# from ...
# import ...

## Descarga del dataset
Se utilizan los mismos datos que en la Tarea 1. Ejecute la siguiente celda para descargar los datos y cargarlos en un dataframe de pandas.

In [ ]:
ds = load_dataset("tomas-gr/all-the-news-2-1-Component-one-sampled", split="train", cache_dir="../data")
df = ds.to_pandas()

# Parte 1: Dataset y representación numérica de texto

## 1. Preparación del dataset
Se utilizará un conjunto de datos reducido de los **tres medios de prensa con mayor cantidad de artículos**.
Se espera que utilice su propia versión de la función `clean_text()` de la Tarea 1.

Particione los datos para generar un conjunto de test del 30% del total, utilizando muestreo estratificado.

**Sugerencia**: utilice el parámetro `stratify` de la función `train_test_split` de scikit-learn y fije también el valor de `random_state` para obtener resultados reproducibles.

In [ ]:
def clean_text(df, column_name):

    # Eliminar primeras palabras hasta el primer "\n"
    result = df[column_name].str.replace(r"^[^\n]*\n", "", regex=True)

    # Convertir todo a minúsculas
    result = result.str.lower()

    # Handle common contractions before removing punctuation
    result = result.str.replace(r"I'm", "I am", regex=True)
    result = result.str.replace(r"you're", "you are", regex=True)
    result = result.str.replace(r"he's", "he is", regex=True)
    result = result.str.replace(r"she's", "she is", regex=True)
    result = result.str.replace(r"it's", "it is", regex=True)
    result = result.str.replace(r"we're", "we are", regex=True)
    result = result.str.replace(r"they're", "they are", regex=True)
    result = result.str.replace(r"I've", "I have", regex=True)
    result = result.str.replace(r"you've", "you have", regex=True)
    result = result.str.replace(r"we've", "we have", regex=True)
    result = result.str.replace(r"they've", "they have", regex=True)
    result = result.str.replace(r"can't", "cannot", regex=True)
    result = result.str.replace(r"won't", "will not", regex=True)
    result = result.str.replace(r"don't", "do not", regex=True)
    result = result.str.replace(r"doesn't", "does not", regex=True)

    # TODO: completar signos de puntuación faltantes (removed "'" since handled above)
    for punc in ["[", "\n", ",", ":", "?", "!", "(", ")", '"', "]"]:
        result = result.str.replace(punc, " ")

    return result

In [ ]:
# TODO: Obtenga los tres medios con mayor cantidad de artículos y filtre el DataFrame
top_3_publications = df['publication'].dropna().value_counts().head(3).index
df_top_3 = df[df['publication'].isin(top_3_publications)]
print('Los 3 medios con más artículos son:')
print(top_3_publications)

# Aplique clean_text sobre la columna de texto elegida y cree una nueva columna "CleanText"
df_top_3["CleanText"] = clean_text(df_top_3, "article")

In [ ]:
# TODO: Particione los datos en train y test (30% test), con muestreo estratificado
X = df_top_3["CleanText"]
y = df_top_3["publication"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)



## 2. Verificación del balance de clases
Genere una visualización que permita verificar que el balance de artículos de cada medio es similar en train y test.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Combine y_train and y_test for easier plotting
# First, convert them to DataFrames or Series to add a 'Set' column
y_train_df = y_train.to_frame(name='publication')
y_train_df['Set'] = 'Train'

y_test_df = y_test.to_frame(name='publication')
y_test_df['Set'] = 'Test'

combined_y_df = pd.concat([y_train_df, y_test_df])

# Calculate and print the counts per publication and set
counts_by_publication_and_set = combined_y_df.groupby(['publication', 'Set']).size().unstack(fill_value=0)
print("Cantidad de artículos por publicación en los conjuntos de entrenamiento y prueba:\n")
print(counts_by_publication_and_set)

# Create the count plot for visual verification
plt.figure(figsize=(10, 6))
sns.countplot(data=combined_y_df, x='publication', hue='Set', palette='viridis')
plt.title('Distribución de Publicaciones en Conjuntos de Entrenamiento y Prueba')
plt.xlabel('Publicación')
plt.ylabel('Cantidad de Artículos')
plt.xticks(rotation=45, ha='right')
plt.legend(title='Conjunto')
plt.tight_layout()
plt.show()

## 3. Representación Bag of Words
Transforme el texto del conjunto de entrenamiento a una representación numérica (features) de conteo de palabras (*bag of words*).
Explique brevemente cómo funciona esta técnica y muestre un ejemplo.
En particular, explique el tamaño de la matriz resultante y la razón por la que es una matriz *sparse*.

**Sugerencia**: puede ser útil imaginar qué sucedería con la memoria RAM requerida si no estuviéramos trabajando con un conjunto de datos reducido.

In [ ]:
# Initialize CountVectorizer
vectorizer = CountVectorizer()

# Handle potential NaN values by filling them with an empty string
X_train_processed = X_train.fillna('')

# Fit and transform X_train_processed
X_train_bow = vectorizer.fit_transform(X_train_processed)

# Show the size of the resulting matrix
print(f"Shape of the Bag of Words matrix: {X_train_bow.shape}")

# Show an example: the first words of the vocabulary and their counts in a document
# Let's take the first document from X_train for example
first_document_index = 0
first_document_bow = X_train_bow[first_document_index].toarray()

# Get feature names (vocabulary)
vocabulary = vectorizer.get_feature_names_out()

# Get words and their counts for the first document
document_word_counts = {}
for word_index in first_document_bow.nonzero()[1]:
    word = vocabulary[word_index]
    count = first_document_bow[0, word_index]
    document_word_counts[word] = count

print(f"\nExample from document {first_document_index}:")
print(f"Original text: {X_train_processed.iloc[first_document_index][:1000]}...") # Show first 200 chars
print(f"Word counts (first 20 words): {dict(list(document_word_counts.items())[:20])}")

## 4. Representación TF-IDF
Explique brevemente qué es un **n-grama**.
Obtenga la representación numérica *Term Frequency - Inverse Document Frequency* (TF-IDF).
Explique brevemente en qué consiste esta transformación adicional.

In [ ]:
# Initialize TfidfVectorizer
tfidf_vectorizer = TfidfVectorizer()

# Handle potential NaN values by filling them with an empty string
X_train_processed = X_train.fillna('')

# Fit and transform X_train_processed
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train_processed)

# Show the size of the resulting matrix
print(f"Shape of the TF-IDF matrix: {X_train_tfidf.shape}")

In [ ]:
first_document_index = 0

# Get the original text of the first document
original_text_first_doc = X_train_processed.iloc[first_document_index]

# Get the feature names (vocabulary) from the TF-IDF vectorizer
vocabulary = tfidf_vectorizer.get_feature_names_out()

# Get the indices and scores of non-zero TF-IDF values for the first document
non_zero_indices = X_train_tfidf[first_document_index].nonzero()[1]
non_zero_scores = X_train_tfidf[first_document_index, non_zero_indices].toarray().flatten()

# Create a list of (word, score) tuples
word_score_pairs = []
for i, idx in enumerate(non_zero_indices):
    word = vocabulary[idx]
    score = non_zero_scores[i]
    word_score_pairs.append((word, score))

# Sort the pairs by score in descending order
word_score_pairs.sort(key=lambda x: x[1], reverse=True)

print(f"\nTop 20 TF-IDF values for document {first_document_index}:")
# Display the top 20 words and their TF-IDF scores
for i, (word, score) in enumerate(word_score_pairs):
    if i >= 20:
        break
    print(f"  {word}: {score:.4f}")

## 5. Visualización PCA sobre TF-IDF
Muestre en un mapa el conjunto de entrenamiento, utilizando las dos primeras componentes PCA sobre los vectores de TF-IDF.
Analice los resultados y compare qué sucede si utiliza:
- a) el filtrado de `stop_words` para idioma inglés;
- b) el parámetro `use_idf=True`;
- c) `ngram_range=(1,2)`.

Opcionalmente, también puede analizar qué sucede si no elimina los signos de puntuación.

¿Se pueden separar los medios de prensa utilizando sólo 2 componentes principales?
Haga una visualización que permita entender cómo varía la varianza explicada a medida que se agregan componentes (por ejemplo, hasta 10 componentes).

Discuta además si la separación observada puede deberse a diferencias de estilo editorial, a diferencias temáticas o a pistas explícitas del medio que no hayan sido removidas en la limpieza.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
import pandas as pd

# TODO: Aplique PCA con 3 componentes sobre X_train_tfidf y grafique los resultados,
# coloreando los puntos según el medio de prensa.

# Initialize PCA with 3 components
pca = PCA(n_components=3, random_state=42)

# Fit and transform X_train_tfidf
X_train_pca = pca.fit_transform(X_train_tfidf.toarray()) # Convert sparse matrix to dense array for PCA

# Create a DataFrame for plotting
pca_df = pd.DataFrame(
    data=X_train_pca,
    columns=['PC1', 'PC2', 'PC3']
)

# Add the publication labels to the DataFrame
pca_df['publication'] = y_train.reset_index(drop=True)

# Plot the results for different component pairs
plt.figure(figsize=(18, 6))

# Plot PC1 vs PC2
plt.subplot(1, 3, 1)
sns.scatterplot(
    x='PC1',
    y='PC2',
    hue='publication',
    data=pca_df,
    palette='viridis',
    alpha=0.7
)
plt.title('PCA of TF-IDF Vectors (PC1 vs PC2) by Publication')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.legend(title='Publication', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True)

# Plot PC1 vs PC3
plt.subplot(1, 3, 2)
sns.scatterplot(
    x='PC1',
    y='PC3',
    hue='publication',
    data=pca_df,
    palette='viridis',
    alpha=0.7
)
plt.title('PCA of TF-IDF Vectors (PC1 vs PC3) by Publication')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 3')
plt.legend(title='Publication', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True)

# Plot PC2 vs PC3
plt.subplot(1, 3, 3)
sns.scatterplot(
    x='PC2',
    y='PC3',
    hue='publication',
    data=pca_df,
    palette='viridis',
    alpha=0.7
)
plt.title('PCA of TF-IDF Vectors (PC2 vs PC3) by Publication')
plt.xlabel('Principal Component 2')
plt.ylabel('Principal Component 3')
plt.legend(title='Publication', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Re-graficar los resultados de PCA con una paleta de colores diferente
# Plot the results for different component pairs with a different palette
plt.figure(figsize=(18, 6))

# Plot PC1 vs PC2
plt.subplot(1, 3, 1)
sns.scatterplot(
    x='PC1',
    y='PC2',
    hue='publication',
    data=pca_df,
    palette='tab10', # Usando una paleta diferente para mayor contraste
    alpha=0.7
)
plt.title('PCA de Vectores TF-IDF (PC1 vs PC2) por Publicación - Paleta Diferente')
plt.xlabel('Componente Principal 1')
plt.ylabel('Componente Principal 2')
plt.legend(title='Publicación', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True)

# Plot PC1 vs PC3
plt.subplot(1, 3, 2)
sns.scatterplot(
    x='PC1',
    y='PC3',
    hue='publication',
    data=pca_df,
    palette='tab10',
    alpha=0.7
)
plt.title('PCA de Vectores TF-IDF (PC1 vs PC3) por Publicación - Paleta Diferente')
plt.xlabel('Componente Principal 1')
plt.ylabel('Componente Principal 3')
plt.legend(title='Publicación', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True)

# Plot PC2 vs PC3
plt.subplot(1, 3, 3)
sns.scatterplot(
    x='PC2',
    y='PC3',
    hue='publication',
    data=pca_df,
    palette='tab10',
    alpha=0.7
)
plt.title('PCA de Vectores TF-IDF (PC2 vs PC3) por Publicación - Paleta Diferente')
plt.xlabel('Componente Principal 2')
plt.ylabel('Componente Principal 3')
plt.legend(title='Publicación', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
import pandas as pd

# Initialize PCA with 3 components
pca_3_components = PCA(n_components=3, random_state=42)

# Fit and transform X_train_tfidf
X_train_pca_3 = pca_3_components.fit_transform(X_train_tfidf.toarray()) # Convert sparse matrix to dense array for PCA

# Create a DataFrame for plotting with 3 components
pca_df_3 = pd.DataFrame(
    data=X_train_pca_3,
    columns=['PC1', 'PC2', 'PC3']
)

# Add the publication labels to the DataFrame
pca_df_3['publication'] = y_train.reset_index(drop=True)

# Plot the results for different component pairs
plt.figure(figsize=(18, 6))

# Plot PC1 vs PC2
plt.subplot(1, 3, 1)
sns.scatterplot(
    x='PC1',
    y='PC2',
    hue='publication',
    data=pca_df_3,
    palette='viridis',
    alpha=0.7
)
plt.title('PCA of TF-IDF Vectors (PC1 vs PC2) by Publication (3 Components)')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.legend(title='Publication', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True)

# Plot PC1 vs PC3
plt.subplot(1, 3, 2)
sns.scatterplot(
    x='PC1',
    y='PC3',
    hue='publication',
    data=pca_df_3,
    palette='viridis',
    alpha=0.7
)
plt.title('PCA of TF-IDF Vectors (PC1 vs PC3) by Publication (3 Components)')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 3')
plt.legend(title='Publication', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True)

# Plot PC2 vs PC3
plt.subplot(1, 3, 3)
sns.scatterplot(
    x='PC2',
    y='PC3',
    hue='publication',
    data=pca_df_3,
    palette='viridis',
    alpha=0.7
)
plt.title('PCA of TF-IDF Vectors (PC2 vs PC3) by Publication (3 Components)')
plt.xlabel('Principal Component 2')
plt.ylabel('Principal Component 3')
plt.legend(title='Publication', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# TODO: Compare los resultados de PCA con distintas configuraciones del TfidfVectorizer:


In [ ]:
# TODO: Genere una visualización que muestre cómo varía la varianza explicada
# a medida que se agregan componentes PCA (por ejemplo, hasta 10 componentes).



# Parte 2: Entrenamiento y Evaluación de Modelos

## 1. Multinomial Naive Bayes
Entrene el modelo *Multinomial Naive Bayes* para clasificar los artículos según a qué medio de prensa pertenece el texto.
Utilice dicho modelo para clasificar los artículos del conjunto de test, y reporte el valor de *accuracy* y la **matriz de confusión**.
Reporte además el valor de *precision* y *recall* para cada medio.
Explique cómo se relacionan estos valores con la matriz anterior.

¿Qué problemas puede tener el hecho de mirar solamente el valor de *accuracy*?
Considere qué sucedería con esta métrica si el desbalance de datos fuera aún mayor entre medios.

**Sugerencia**: utilice el método `from_predictions` de `ConfusionMatrixDisplay` para realizar la matriz.

In [ ]:
# TODO: Entrene Multinomial Naive Bayes sobre X_train_tfidf


In [ ]:
# TODO: Evalúe el modelo sobre el conjunto de test


## 2. Validación cruzada y búsqueda de hiperparámetros
Explique cómo funciona la técnica de **validación cruzada** (*cross-validation*).
Implemente una búsqueda de hiperparámetros usando `GridSearchCV`.
Genere una visualización que permita comparar las métricas (por ejemplo, *accuracy*) de los distintos modelos entrenados, viendo el valor promedio y la variabilidad de las mismas en todos los *splits* (por ejemplo, en un gráfico de violín).

In [ ]:
# TODO: Defina la grilla de hiperparámetros y ejecute GridSearchCV


In [ ]:
# TODO: Genere una visualización (por ejemplo, gráfico de violín) que compare
# la accuracy de los distintos modelos entrenados en GridSearchCV,
# mostrando la variabilidad en los distintos splits.


## 3. Entrenamiento final con el mejor modelo
Elija el mejor modelo (mejores parámetros) y vuelva a entrenar sobre todo el conjunto de entrenamiento disponible (sin quitar datos para validación).
Reporte el valor final de las métricas y la matriz de confusión.
Discuta las limitaciones de utilizar un modelo basado en *bag of words* o TF-IDF para el análisis de texto.

In [ ]:
# TODO: Entrene el mejor modelo sobre todo el conjunto de entrenamiento


In [ ]:
# TODO: Evalúe el mejor modelo sobre el conjunto de test y reporte métricas finales


## 4. Modelo alternativo
Evalúe al menos un modelo más (dentro de scikit-learn) aparte de *Multinomial Naive Bayes* para clasificar el texto utilizando las mismas *features* de texto.
Explique brevemente cómo funciona y compare los resultados con el anterior.

In [ ]:
# TODO: Entrene al menos un modelo alternativo de scikit-learn


## 5. Cambio de medio de prensa
Evalúe el problema cambiando al menos un medio de prensa.
En particular, observe el (des)balance de datos y los problemas que pueda generar, así como cualquier indicio que pueda ver en el mapeo previo con PCA.
Puede ser útil comentar acerca de técnicas como sobre-muestreo y submuestreo; no es necesario implementarlas.

In [ ]:
# TODO: Seleccione una combinación diferente de tres medios de prensa (cambiando al menos uno)
# y repita el proceso de entrenamiento y evaluación.

# Analice el balance de clases en este nuevo conjunto de datos
# ...

## 6. Técnica alternativa de extracción de features
Busque información sobre al menos una técnica alternativa de extraer *features* de texto.
Explique brevemente cómo funciona y qué tipo de diferencias esperaría en los resultados.
No se espera que implemente nada en esta parte.

*TODO: Escriba su análisis en el informe.*

## 7. Opcional: Clasificación a nivel de título
Repita la clasificación con los tres medios de prensa originales, pero esta vez clasificando a nivel de **título** en lugar de artículo completo.

In [ ]:
# Opcional: Repita el pipeline de clasificación utilizando el título del artículo
# en lugar del cuerpo del texto.
